In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, DataCollatorForSeq2Seq
from peft import LoraConfig, get_peft_model
from datasets import load_dataset
import transformers

In [ ]:
model_name = "mistralai/Mistral-7B-Instruct-v0.2"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, device_map="auto")

lora = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora)

In [ ]:
dataset = load_dataset("json", data_files={
    "train": "../data/sft_train.json",
    "val": "../data/sft_val.json"
})

def preprocess(batch):
    text = "Instruction: " + batch["instruction"] + "\nInput: " + batch["input"] + "\nAnswer: " + batch["output"]
    batch["input_ids"] = tokenizer(text, truncation=True, padding="max_length", max_length=1024).input_ids
    return batch

dataset = dataset.map(preprocess)

In [ ]:
args = TrainingArguments(
    output_dir="../model/lora_output",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    num_train_epochs=3,
    bf16=True,
    logging_steps=50,
    save_steps=500,
    evaluation_strategy="steps",
    eval_steps=200,
)

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

trainer = transformers.Trainer(
    model=model,
    args=args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["val"],
    data_collator=data_collator
)

trainer.train()
model.save_pretrained("../model/final_lora_weights")